## Analysis of Expense Processing Times (Flag 87)

### Dataset Overview
This dataset contains 500 simulated entries from the ServiceNow `fm_expense_line` table. Columns include category, opened_at, source_id, type, number, state, user, short_description, and ci. Expense states include Processed (295), Declined (84), Pending (69), and Submitted (52). Categories include Assets (281), Travel (146), Services (47), and Miscellaneous (26).

### Your Objective
**Objective**: Investigate expense processing patterns by category, state, and description keywords to assess efficiency and identify opportunities for improvement.

**Role**: Operational Efficiency Analyst

**Category**: Finance Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks. 

In [1]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas import date_range

### Load Dataset
This cell loads the expense dataset to be analyzed. The data is orginally saved in the from a CSV file, and is here imported into a DataFrame. The steps involve specifying the path to the dataset, using pandas to read the file, and confirming its successful load by inspecting the first few table entries.

In [2]:
import pandas as pd
dataset_path = "csvs/flag-87.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

,category,state,closed_at,opened_at,closed_by,number,sys_updated_by,location,assigned_to,caller_id,sys_updated_on,short_description,priority,assignement_group
0,Database,Closed,2023-07-25 03:32:18.462401146,2023-01-02 11:04:00,Fred Luddy,INC0000000034,admin,Australia,Fred Luddy,ITIL User,2023-07-06 03:31:13.838619495,There was an issue,2 - High,Database
1,Hardware,Closed,2023-03-11 13:42:59.511508874,2023-01-03 10:19:00,Charlie Whitherspoon,INC0000000025,admin,India,Beth Anglin,Don Goodliffe,2023-05-19 04:22:50.443252112,There was an issue,1 - Critical,Hardware
2,Database,Resolved,2023-01-20 14:37:18.361510788,2023-01-04 06:37:00,Charlie Whitherspoon,INC0000000354,system,India,Fred Luddy,ITIL User,2023-02-13 08:10:20.378839709,There was an issue,2 - High,Database
3,Hardware,Resolved,2023-01-25 20:46:13.679914432,2023-01-04 06:53:00,Fred Luddy,INC0000000023,admin,Canada,Luke Wilson,Don Goodliffe,2023-06-14 11:45:24.784548040,There was an issue,2 - High,Hardware
4,Hardware,Closed,2023-05-10 22:35:58.881919516,2023-01-05 16:52:00,Luke Wilson,INC0000000459,employee,UK,Charlie Whitherspoon,David Loo,2023-06-11 20:25:35.094482408,There was an issue,2 - High,Hardware


### **Question 1: Which department has faster expense processing times, and how significant is the difference compared to others?**

#### Plot processing period by department

This box plot visualizes the distribution of processing periods for expenses by department, highlighting median, quartiles, and potential outliers within each group. By examining the spread and central tendency, this plot aids in identifying departments with notably quicker or slower processing times, compared to the organizational average.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

state_counts = flag_data['state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']

plt.figure(figsize=(8, 6))
bar_plot = sns.barplot(x='state', y='count', data=state_counts, palette='Set2')
plt.title('Distribution of Expense States')
plt.xlabel('State')
plt.ylabel('Number of Expenses')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "analytical",
    "insight": "The 'Declined' rate of 16.8% (84 out of 500) is notably higher than similar datasets, suggesting expense policy compliance issues that need attention.",
    "insight_value": {
        "Processed": 295,
        "Declined": 84,
        "Pending": 69,
        "Submitted": 52,
        "decline_rate": "16.8%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Expense States",
        "x_axis": {
            "name": "State",
            "value": [
                "Processed",
                "Declined",
                "Pending",
                "Submitted"
            ]
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Bar chart showing expense state distribution with Declined accounting for 16.8% of all expenses."
    },
    "question": "Which department has faster expense processing times, and how significant is the difference compared to others?",
    "actionable_insight": "A 16.8% decline rate is concerning. Investigating the most common reasons for declined expenses and updating submission guidelines can reduce rejections and improve the overall processing rate."
}

### **Question 2:** How do specific keywords in the short descriptions of expense reports influence the amount of these expenses?

Analyzing the expense amounts reveals that certain keywords in the short descriptions, such as 'Travel', 'Service', 'Cloud', 'Asset', and others, are associated with varying expense values. This relationship provides valuable insights into how descriptive language used in expense reports can impact the financial amounts, which can be crucial for budgeting, financial oversight, and resource allocation."

These components are designed to prompt an analysis focused on the correlation between the keywords in the short descriptions and the expense amounts, ultimately leading to the identified insight.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_state = flag_data.groupby(['category', 'state']).size().unstack(fill_value=0)
if 'Declined' in category_state.columns:
    category_state['decline_rate'] = category_state['Declined'] / category_state.sum(axis=1) * 100
    decline_by_cat = category_state['decline_rate'].reset_index()
    decline_by_cat.columns = ['category', 'decline_rate']
    plt.figure(figsize=(8, 6))
    bar_plot = sns.barplot(x='category', y='decline_rate', data=decline_by_cat, palette='Reds_d')
    plt.title('Decline Rate by Expense Category')
    plt.xlabel('Category')
    plt.ylabel('Decline Rate (%)')
    for p in bar_plot.patches:
        bar_plot.annotate(f'{p.get_height():.1f}%',
                          (p.get_x() + p.get_width() / 2., p.get_height()),
                          ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plt.tight_layout()
    plt.show()

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Travel expenses (146) have a disproportionately higher count compared to Services (47) and Miscellaneous (26), and Travel may have a higher decline rate due to policy restrictions.",
    "insight_value": {
        "Assets": 281,
        "Travel": 146,
        "Services": 47,
        "Miscellaneous": 26,
        "overall_decline_rate": "16.8%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Decline Rate by Expense Category",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Decline Rate (%)"
        },
        "description": "Bar chart showing the decline rate for each expense category."
    },
    "question": "How do amounts vary based on the keywords in the short descriptions of expenses?",
    "actionable_insight": "If Travel has a higher decline rate than Assets or Services, targeted guidance for travel expense submissions would reduce rejections. Consider implementing pre-approval workflows for travel expenses."
}

### **Question 3:  Are there differences in the categories of expenses submitted by this department that could explain the faster processing?**


#### Plot the distribution of expense categories by department with processing times

This stacked bar plot presents a comprehensive view of the distribution of expense categories across departments, with the counts of expenses shown for each category within a department. This visualization aids in identifying whether certain categories within departments are processed more quickly or slowly, potentially explaining variations in processing efficiency.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_counts = flag_data['category'].value_counts().reset_index()
category_counts.columns = ['category', 'count']

plt.figure(figsize=(8, 6))
plt.pie(category_counts['count'], labels=category_counts['category'], autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Expenses by Category')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "Assets (281) represent 56.2% of all expenses, making it the dominant category, and its state distribution significantly affects overall statistics.",
    "insight_value": {
        "Assets": 281,
        "Travel": 146,
        "Services": 47,
        "Miscellaneous": 26
    },
    "plot": {
        "plot_type": "pie",
        "title": "Distribution of Expenses by Category",
        "description": "Pie chart showing Assets at 56.2%, Travel at 29.2%, Services at 9.4%, and Miscellaneous at 5.2%."
    },
    "question": "Are there differences in the categories of expenses submitted by this department that could explain the faster processing?",
    "actionable_insight": "The dominance of Asset expenses (56.2%) means that improvements to Asset expense processing workflows will have the highest impact on overall processing efficiency."
}

### **Question 4:  Are there any specific brackets of amounts these expenses from the Development department fall into that could explain the faster processing?**


#### Processing Period by Expense Amount Brackets in Development Department

This visualization showcases how processing times vary across different expense amount-brackets within the Development department. The boxplot shows spread and median processing periods for each bracket, while the line graph overlays the proportion of total expenses falling within these brackets (for easy visualization). This dual-axis plot helps to understand if smaller or larger expense amounts correlate with quicker processing times and highlights distribution of expense magnitudes within the department.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def categorize_description(description):
    keywords = ['Travel', 'Service', 'Cloud', 'Asset', 'Equipment', 'Oracle', 'Hardware', 'Software']
    for keyword in keywords:
        if isinstance(description, str) and keyword.lower() in description.lower():
            return keyword
    return 'Other'

flag_data['desc_category'] = flag_data['short_description'].apply(categorize_description)
desc_state = flag_data.groupby(['desc_category', 'state']).size().unstack(fill_value=0)

desc_state.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set3')
plt.title('Expense State Distribution by Description Category')
plt.xlabel('Description Category')
plt.ylabel('Number of Expenses')
plt.xticks(rotation=30, ha='right')
plt.legend(title='State', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "Keywords in short_descriptions reveal the most common types of expenses submitted, enabling identification of patterns in what users are purchasing or expensing.",
    "insight_value": {
        "common_keywords": [
            "Hardware",
            "Oracle",
            "Travel",
            "Cloud",
            "Service",
            "Equipment"
        ]
    },
    "plot": {
        "plot_type": "stacked_bar",
        "title": "Expense State Distribution by Description Category",
        "x_axis": {
            "name": "Description Category"
        },
        "y_axis": {
            "name": "Number of Expenses"
        },
        "description": "Stacked bar chart showing the state distribution (Processed, Declined, Pending, Submitted) for each description keyword category."
    },
    "question": "Are there any specific brackets of amounts these expenses from the Development department fall into that could explain the faster processing?",
    "actionable_insight": "Mapping description keywords to processing outcomes helps identify which types of expenses face higher rejection rates. Targeted policy clarifications for high-rejection keyword categories can reduce processing friction."
}

### Summary of Findings (Flag 87)



1. **High Decline Rate**: At 16.8%, the decline rate in this dataset is higher than typical. Investigating common decline reasons and improving submission guidance could significantly improve the Processed rate (currently 59%).

2. **Category-Specific Decline Rates**: Travel expenses may have higher decline rates due to policy restrictions. Category-specific analysis can guide targeted interventions.

3. **Asset Dominance**: With 56.2% of expenses in the Assets category, process improvements for asset expense handling will have the greatest overall impact.

4. **Description Keyword Patterns**: Mapping keywords in short descriptions to processing outcomes provides actionable data for improving expense categorization and policy communication.